# NB8 — Output-head width ablation

Goal: test whether increasing the final `output_mlp` hidden width above the canonical V5 value `16` improves validation ROC-AUC.

Variants: `32`, `64`, `128`. Everything else stays on the frozen V5 recipe: BCE-only, FP32, LR `3e-4`, batch size `256`, max 60 epochs, patience 10, min 30 epochs, seed 42, checkpoint selection by validation ROC-AUC.

Important fairness detail: changing `output_hidden_dim` changes RNG consumption before the category embedding re-init. This notebook therefore uses `src/scorer/head_width_experiment.py` to copy the canonical width-16 initialization for `category_embedding`, `item_mlp`, and `pair_mlp`; only the output head starts width-specific. All parameters are then trained normally.

This is an experiment branch only. It does not overwrite frozen V5 on `main`.


In [ ]:
from pathlib import Path
import copy, json, os, subprocess, sys

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
BRANCH = "exp/scorer-output-head-width"
REPO_ROOT = Path("/content/opisoverated")

if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", BRANCH], check=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from google.colab import drive
drive.mount("/content/drive")

# Read frozen dataset/embedding artifacts from ML_Final.
ARTIFACT_ROOT = Path("/content/drive/MyDrive/ML_Final")
os.environ["FASHION_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["FASHION_EMBEDDING_CACHE"] = str(ARTIFACT_ROOT / "fashionclip_item_embeddings.pt")
os.environ["FASHION_EMBEDDING_MANIFEST"] = str(ARTIFACT_ROOT / "embedding_manifest_v1.json")
os.environ["FASHION_CORE7_DIR"] = str(ARTIFACT_ROOT / "polyvore_core7_v2" / "core7_drop_v2")
os.environ["FASHION_SCORER_READY_DIR"] = str(ARTIFACT_ROOT / "polyvore_core7_v2" / "scorer_ready_v2")

# Save experiment checkpoints in your own MyDrive root to avoid shared-folder write issues.
OUTPUT_ROOT = Path("/content/drive/MyDrive/scorer_head_width_runs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"], check=True)
HEAD = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
print("Branch:", BRANCH)
print("Git HEAD:", HEAD)
print("Output root:", OUTPUT_ROOT)


In [ ]:
import yaml, torch

from src.data.runtime_paths import load_runtime_paths
from src.scorer.checkpoint import build_runtime_provenance, load_checkpoint
from src.scorer.head_width_experiment import build_output_head_width_variant
from src.scorer.train import build_train_valid_loaders, evaluate_epoch, fit_scorer, validate_s3_config

BASE_CONFIG_PATH = REPO_ROOT / "configs/scorer_type_aware_pairwise_v1_val_auc.yaml"
EXP_CONFIG_PATH = REPO_ROOT / "configs/scorer_output_head_width_ablation.yaml"
with BASE_CONFIG_PATH.open("r", encoding="utf-8") as f:
    base_config = yaml.safe_load(f)
with EXP_CONFIG_PATH.open("r", encoding="utf-8") as f:
    exp_config = yaml.safe_load(f)

validate_s3_config(base_config)
WIDTHS = [int(x) for x in exp_config["experiment"]["widths"]]
SEED = int(exp_config["experiment"]["seed"])
print("Widths to train:", WIDTHS)
print("Canonical comparison: width=16, AUC=0.6905082489625538, FITB=0.7626970227670753")


In [ ]:
paths = load_runtime_paths(repo_root=REPO_ROOT)
provenance = build_runtime_provenance(paths, REPO_ROOT)
assert provenance["git_tree_clean"] is True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "Switch Colab to a GPU runtime. Training remains FP32."

# Smoke-check dataset once. Each actual run rebuilds fresh loaders so shuffled
# sample order starts from the same DataLoader-local seed.
smoke = build_train_valid_loaders(paths, base_config, num_workers=0)
assert len(smoke["datasets"]["train"]) == 30918
assert len(smoke["datasets"]["valid"]) == 2284
del smoke
print("DATA / PROVENANCE / GPU: PASS")


In [ ]:
results = [{
    "output_hidden_dim": 16,
    "best_epoch": 52,
    "valid_roc_auc": 0.6905082489625538,
    "valid_fitb_2way": 0.7626970227670753,
    "source": "frozen_v5_main",
}]

for width in WIDTHS:
    print("\n" + "=" * 72)
    print(f"TRAINING output_hidden_dim={width}")
    print("=" * 72)

    config = copy.deepcopy(base_config)
    config["model"]["output_hidden_dim"] = int(width)
    validate_s3_config(config)

    # Fresh loaders are important: identical shuffle RNG start per ablation.
    loaders = build_train_valid_loaders(paths, config, num_workers=0)
    train_loader = loaders["train_loader"]
    valid_loader = loaders["valid_loader"]

    model = build_output_head_width_variant(
        config, output_hidden_dim=width, seed=SEED
    ).to(device)
    assert model.output_hidden_dim == width

    run_dir = OUTPUT_ROOT / f"output_hidden_{width}_seed{SEED}"
    if run_dir.exists() and any(run_dir.iterdir()):
        raise RuntimeError(
            f"Run already exists: {run_dir}. Rename/delete it intentionally before rerunning."
        )

    fit_result = fit_scorer(
        model, train_loader, valid_loader,
        config=config, checkpoint_dir=run_dir, provenance=provenance, device=device,
    )

    best_path = run_dir / "best.pt"
    best_model = build_output_head_width_variant(
        config, output_hidden_dim=width, seed=SEED
    ).to(device)
    payload = load_checkpoint(
        best_path, model=best_model, map_location=device, current_provenance=provenance
    )
    best_model.eval()
    criterion = torch.nn.BCEWithLogitsLoss()
    metrics = evaluate_epoch(best_model, valid_loader, criterion=criterion, device=device)

    row = {
        "output_hidden_dim": width,
        "best_epoch": int(payload["epoch"]),
        "valid_roc_auc": float(metrics["roc_auc"]),
        "valid_fitb_2way": float(metrics["fitb_2way"]),
        "mean_logit_margin": float(metrics["mean_logit_margin"]),
        "median_logit_margin": float(metrics["median_logit_margin"]),
        "valid_loss": float(metrics["loss"]),
        "source": "head_width_ablation",
    }
    results.append(row)
    print("RESULT:", json.dumps(row, indent=2))

    # Free GPU memory before next width.
    del model, best_model, loaders, train_loader, valid_loader
    torch.cuda.empty_cache()


In [ ]:
print("\nFINAL COMPARISON")
print(f"{'width':>8} {'epoch':>8} {'AUC':>10} {'FITB':>10} {'delta_AUC':>12}")
baseline_auc = results[0]["valid_roc_auc"]
for row in results:
    delta = row["valid_roc_auc"] - baseline_auc
    print(f"{row['output_hidden_dim']:>8} {row['best_epoch']:>8} {row['valid_roc_auc']:>10.6f} {row['valid_fitb_2way']:>10.6f} {delta:>+12.6f}")

best = max(results, key=lambda r: r["valid_roc_auc"])
print("\nBest by validation ROC-AUC:", best)

SUMMARY_PATH = OUTPUT_ROOT / "head_width_ablation_summary.json"
SUMMARY_PATH.write_text(json.dumps({"git_head": HEAD, "results": results}, indent=2), encoding="utf-8")
print("Saved summary:", SUMMARY_PATH)
print("TEST SPLIT WAS NOT LOADED.")
